In [5]:
import time
import re
import os
import random
import pandas as pd
import chromedriver_autoinstaller

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    TimeoutException,
    NoAlertPresentException,
    UnexpectedAlertPresentException,
    NoSuchElementException,
)

# =========================
# 🔧 0. 경로 & 기본 설정
# =========================

EXCEL_PATH = "yeoshinticket_event_urls.xlsx"  # URL 목록 엑셀

# ✅ 출력 파일
OUTPUT_CSV = "yeoti_detail_procedure_part_01.csv"
ERROR_LOG_CSV = "yeoti_detail_procedure_errors_01.csv"

# ✅ 테스트 / 재시도 설정
MAX_ROWS = 100        # 전체 돌릴 땐 None, 일부만 돌리고 싶으면 숫자
SAVE_INTERVAL = 10    # N건마다 중간 저장
MAX_RETRY = 3         # 페이지 재시도 횟수

# =========================
# 🔍 테스트 크롤링 옵션
# =========================
TEST_MODE = True           # 랜덤 테스트 크롤링 모드
TEST_SAMPLE_SIZE = 100     # TEST_MODE=True일 때 크롤링할 랜덤 URL 개수

BASE_URL = "https://www.yeoshin.co.kr"

# =========================
# 1. 기본 셀렉터
# =========================

# 시술명
SEL_TREATMENT_NAME = (
    "#ct-view > div.relative.w-full > div.relative.bg-white > "
    "div.px-\\[16px\\].relative.bg-white.w-full > "
    "div.flex.flex-col.justify-center.w-full.py-\\[16px\\] > "
    "section.flex.flex-col.gap-\\[4px\\] > h1"
)

# 메인 이미지
SEL_MAIN_IMAGE = (
    "#ct-view > div.relative.w-full > div.relative.overflow-hidden > div > img"
)

# 평점 / 후기수
SEL_RATING = (
    "#ct-view > div.relative.w-full > div.relative.bg-white > "
    "div.px-\\[16px\\].relative.bg-white.w-full > "
    "div.flex.flex-col.justify-center.w-full.py-\\[16px\\] > "
    "section.flex.flex-col.gap-\\[4px\\] > button > div > span"
)

SEL_REVIEW_COUNT = (
    "#ct-view > div.relative.w-full > div.relative.bg-white > "
    "div.px-\\[16px\\].relative.bg-white.w-full > "
    "div.flex.flex-col.justify-center.w-full.py-\\[16px\\] > "
    "section.flex.flex-col.gap-\\[4px\\] > button > span"
)

# 가격 정보 (정상가)
SEL_ORIGINAL_PRICE = (
    "#ct-view > div.relative.w-full > div.relative.bg-white > "
    "div.px-\\[16px\\].relative.bg-white.w-full > "
    "div.flex.flex-col.justify-center.w-full.py-\\[16px\\] > "
    "section.flex.items-end.justify-between.w-full.mt-\\[8px\\] > "
    "div > div:nth-child(1) > div"
)

# 판매가 (할인 있는 경우: div > div > h2)
SEL_SELLING_PRICE_DISCOUNT = (
    "#ct-view > div.relative.w-full > div.relative.bg-white > "
    "div.px-\\[16px\\].relative.bg-white.w-full > "
    "div.flex.flex-col.justify-center.w-full.py-\\[16px\\] > "
    "section.flex.items-end.justify-between.w-full.mt-\\[8px\\] > "
    "div > div > h2"
)

# 판매가 (할인 없는 경우: div > h2)
SEL_SELLING_PRICE_NO_DISCOUNT = (
    "#ct-view > div.relative.w-full > div.relative.bg-white > "
    "div.px-\\[16px\\].relative.bg-white.w-full > "
    "div.flex.flex-col.justify-center.w-full.py-\\[16px\\] > "
    "section.flex.items-end.justify-between.w-full.mt-\\[8px\\] > "
    "div > h2"
)

# 옵션 설명
SEL_OPTION_INFO = (
    "#ct-view > div.relative.w-full > div.relative.bg-white > "
    "div.px-\\[16px\\].relative.bg-white.w-full > "
    "div.flex.flex-col.justify-center.w-full.py-\\[16px\\] > "
    "section.flex.items-end.justify-between.w-full.mt-\\[8px\\] > "
    "div > div.flex.items-end.gap-\\[4px\\] > "
    "div.text-\\[12px\\].mb-\\[2px\\].font-normal.leading-\\[18px\\].text-gray500"
)

# 병원명
SEL_HOSPITAL_NAME = (
    "#ct-view > div.relative.w-full > div.relative.bg-white > "
    "div.flex.flex-col.mt-\\[32px\\].justify-center.w-full.gap-\\[32px\\] > "
    "article > div > div > h2"
)

# 주소 – 옛날 방식(섹션 인덱스 기반, fallback용)
SEL_HOSPITAL_ADDRESS_OLD = (
    "#ct-view > div.relative.w-full > div.relative.bg-white > "
    "div.flex.flex-col.mt-\\[32px\\].justify-center.w-full.gap-\\[32px\\] > "
    "article > section:nth-child(3) > div > div > div"
)

# 시술 설명 (fallback)
SEL_TREATMENT_DESCRIPTION = (
    "#ct-view > div.relative.w-full > div:nth-child(5) > div > div > "
    "div.sc-1543ab3d-0.sc-1543ab3d-1.hQTMVb.eFoKOT > "
    "article:nth-child(1) > section"
)

# 시술시간/마취여부/회복시간/시술효과 묶음
SEL_DOWNTIME_INFO = (
    "#ct-view > div.relative.w-full > div.relative.bg-white > "
    "section.flex.flex-col.px-\\[16px\\].py-\\[24px\\] > div:nth-child(1)"
)

# ====== 타겟/특장점/Q&A/주의사항 fallback 셀렉터 ======

SEL_PROCEDURE_INFO_FALLBACK = (
    "#ct-view > div.relative.w-full > div:nth-child(4) > div > div > "
    "div.sc-1543ab3d-0.sc-1543ab3d-1.hQTMVb.eFoKOT > article.relative > section"
)

SEL_TARGET_GROUP_FALLBACK = (
    "#ct-view > div.relative.w-full > div:nth-child(4) > div > div > "
    "div.sc-1543ab3d-0.sc-1543ab3d-1.hQTMVb.eFoKOT > article:nth-child(32) > section"
)

SEL_PROCEDURE_QNA_FALLBACK = (
    "#ct-view > div.relative.w-full > div:nth-child(4) > div > div > "
    "div.sc-1543ab3d-0.sc-1543ab3d-1.hQTMVb.eFoKOT > "
    "article.pt-\\[32px\\].pb-\\[40px\\].px-\\[24px\\].flex.flex-col.gap-\\[32px\\]"
)

SEL_SIDE_EFFECT_INFO = (
    "#ct-view > div.relative.w-full > div:nth-child(4) > div > div > "
    "div.sc-1543ab3d-0.sc-1543ab3d-1.hQTMVb.eFoKOT > article.p-\\[12px\\] > section"
)

# =========================
# 2. 드라이버 / 유틸 함수
# =========================

def init_driver():
    chromedriver_autoinstaller.install()
    options = Options()
    options.add_argument("--headless=new")  # 디버깅 시 주석 처리
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")

    prefs = {"profile.managed_default_content_settings.images": 2}
    options.add_experimental_option("prefs", prefs)

    driver = webdriver.Chrome(options=options)
    return driver


def safe_text(driver, selector):
    if not selector:
        return None
    try:
        el = driver.find_element(By.CSS_SELECTOR, selector)
        return el.text.strip()
    except Exception:
        return None


def safe_attr(driver, selector, attr):
    if not selector:
        return None
    try:
        el = driver.find_element(By.CSS_SELECTOR, selector)
        val = el.get_attribute(attr)
        return val.strip() if val else None
    except Exception:
        return None


def clean_lines(raw_text):
    if not raw_text:
        return []
    lines = []
    for ln in raw_text.splitlines():
        txt = re.sub(r"\s+", " ", ln).strip()
        if not txt:
            continue
        lines.append(txt)
    if not lines:
        return []
    seen = set()
    uniq = []
    for txt in lines:
        if txt in seen:
            continue
        seen.add(txt)
        uniq.append(txt)
    return uniq


def safe_section_text(driver, selector, sep=" "):
    if not selector:
        return None
    try:
        root = driver.find_element(By.CSS_SELECTOR, selector)
        raw = root.text
    except Exception:
        return None
    uniq_lines = clean_lines(raw)
    return sep.join(uniq_lines) if uniq_lines else None


def section_text_from_element(element, sep=" "):
    try:
        raw = element.text
    except Exception:
        return None
    uniq_lines = clean_lines(raw)
    return sep.join(uniq_lines) if uniq_lines else None


def extract_number(text):
    if not text:
        return None
    digits = re.sub(r"[^\d]", "", text)
    return int(digits) if digits else None


def normalize_price_to_int(text):
    return extract_number(text)


def clean_dataframe_for_excel(df_out: pd.DataFrame) -> pd.DataFrame:
    illegal_pattern = re.compile(r"[\x00-\x08\x0b-\x0c\x0e-\x1f]")
    MAX_LEN = 32000

    def clean_for_excel(val):
        if pd.isna(val):
            return val
        s = str(val)
        s = illegal_pattern.sub("", s)
        if len(s) > MAX_LEN:
            s = s[:MAX_LEN]
        return s

    obj_cols = df_out.select_dtypes(include=["object"]).columns
    for col in obj_cols:
        df_out[col] = df_out[col].apply(clean_for_excel)
    return df_out


def save_results(results, path, final=False):
    if not results:
        if final:
            print("⚠️ 크롤링된 데이터가 없어 빈 CSV/XLSX 파일만 생성했습니다.")
        return

    df_out = pd.DataFrame(results)
    df_out = clean_dataframe_for_excel(df_out)

    df_out.to_csv(path, index=False, encoding="utf-8-sig")

    xlsx_path = os.path.splitext(path)[0] + ".xlsx"
    try:
        with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
            df_out.to_excel(writer, index=False, sheet_name="data")
    except Exception as e:
        print(f"⚠️ XLSX 저장 중 에러 발생: {e}")

    if final:
        print(f"[최종 저장] {len(df_out)}건 → {path}, {xlsx_path}")
    else:
        print(f"[중간 저장] {len(df_out)}건 → {path}, {xlsx_path}")


def log_error_row(url, depth1, depth2, depth3, exc):
    err_info = {
        "event_url": url,
        "대분류": depth1,
        "중분류": depth2,
        "소분류": depth3,
        "error_type": type(exc).__name__,
        "error_msg": str(exc),
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    df_err = pd.DataFrame([err_info])

    if os.path.exists(ERROR_LOG_CSV):
        df_err.to_csv(
            ERROR_LOG_CSV,
            mode="a",
            header=False,
            index=False,
            encoding="utf-8-sig",
        )
    else:
        df_err.to_csv(ERROR_LOG_CSV, index=False, encoding="utf-8-sig")

    try:
        df_all = pd.read_csv(ERROR_LOG_CSV, encoding="utf-8-sig")
        df_all = clean_dataframe_for_excel(df_all)
        err_xlsx = os.path.splitext(ERROR_LOG_CSV)[0] + ".xlsx"
        with pd.ExcelWriter(err_xlsx, engine="openpyxl") as writer:
            df_all.to_excel(writer, index=False, sheet_name="error_log")
    except Exception as e:
        print(f"⚠️ 에러 로그 XLSX 동기화 중 에러 발생: {e}")


# =========================
# 3. 병원/타겟/특장점/Q&A/가격 헬퍼
# =========================

def get_hospital_address(driver):
    try:
        copy_btn = driver.find_element(
            By.XPATH,
            "//button[.//span[contains(normalize-space(.), '주소복사')]]"
        )
        addr_span = copy_btn.find_element(
            By.XPATH,
            "./preceding-sibling::span[1]"
        )
        addr = addr_span.text.strip()
        if addr:
            return addr
    except NoSuchElementException:
        pass
    except Exception:
        pass
    return safe_text(driver, SEL_HOSPITAL_ADDRESS_OLD)


def get_treatment_description(driver):
    try:
        article = driver.find_element(
            By.XPATH,
            "//article[contains(@style,'background-color') and "
            ".//p[contains(@class,'leading-')]]"
        )
        text = section_text_from_element(article, sep=" ")
        if text:
            return text
    except Exception:
        pass
    return safe_section_text(driver, SEL_TREATMENT_DESCRIPTION, sep=" ")


def get_target_group(driver):
    """
    추천대상:
    - article 안에 '추천대상' 텍스트가 있으면 그 article을 잡고
    - p.text-center.break-keep 텍스트들을 모아 하나의 문자열로 반환
    """
    try:
        article = driver.find_element(
            By.XPATH,
            "//article[.//*[contains(normalize-space(.), '추천대상')]]"
        )
        ps = article.find_elements(
            By.XPATH,
            ".//p[contains(@class,'text-center') and contains(@class,'break-keep')]"
        )
        texts = [p.text.strip() for p in ps if p.text.strip()]
        if texts:
            uniq, seen = [], set()
            for t in texts:
                if t in seen:
                    continue
                seen.add(t)
                uniq.append(t)
            return " ".join(uniq) if uniq else None
    except Exception:
        pass

    return safe_section_text(driver, SEL_TARGET_GROUP_FALLBACK, sep=" ")


def get_procedure_info(driver):
    try:
        article = driver.find_element(
            By.XPATH,
            "//article["
            "  .//h2[(contains(normalize-space(.), '시술') and contains(normalize-space(.), '특장점'))]"
            "   or "
            "  .//h3[(contains(normalize-space(.), '시술') and contains(normalize-space(.), '특장점'))]"
            "]"
        )
        text = section_text_from_element(article, sep=" ")
        if text:
            return text
    except Exception:
        pass
    return safe_section_text(driver, SEL_PROCEDURE_INFO_FALLBACK, sep=" ")


def get_procedure_qna(driver):
    try:
        article = driver.find_element(
            By.XPATH,
            "//article["
            "  .//h2[(contains(normalize-space(.), '자주') and contains(normalize-space(.), '질문'))]"
            "   or "
            "  .//h3[(contains(normalize-space(.), '자주') and contains(normalize-space(.), '질문'))]"
            "]"
        )
        return section_text_from_element(article, sep=" ")
    except Exception:
        pass
    return safe_section_text(driver, SEL_PROCEDURE_QNA_FALLBACK, sep=" ")


def get_side_effect_info(driver):
    return safe_section_text(driver, SEL_SIDE_EFFECT_INFO, sep=" ")


def get_price_fields(driver):
    original_raw = safe_text(driver, SEL_ORIGINAL_PRICE)
    selling_raw = safe_text(driver, SEL_SELLING_PRICE_DISCOUNT)
    if not selling_raw:
        selling_raw = safe_text(driver, SEL_SELLING_PRICE_NO_DISCOUNT)
    original_price = normalize_price_to_int(original_raw)
    selling_price = normalize_price_to_int(selling_raw)
    return original_price, selling_price


# =========================
# 4. 단일 페이지 크롤링
# =========================

def crawl_event_page(driver, url, depth1=None, depth2=None, depth3=None):
    driver.get(url)

    WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "#ct-view"))
    )

    treatment_name = safe_text(driver, SEL_TREATMENT_NAME)
    main_image_url = safe_attr(driver, SEL_MAIN_IMAGE, "src")

    rating_text = safe_text(driver, SEL_RATING)
    try:
        rating = float(rating_text) if rating_text else None
    except ValueError:
        rating = None

    review_count_raw = safe_text(driver, SEL_REVIEW_COUNT)
    review_count = extract_number(review_count_raw)

    original_price, selling_price = get_price_fields(driver)
    option_info = safe_text(driver, SEL_OPTION_INFO)

    hospital_name = safe_text(driver, SEL_HOSPITAL_NAME)
    hospital_address = get_hospital_address(driver)

    treatment_description = get_treatment_description(driver)
    procedure_info = get_procedure_info(driver)
    target_group = get_target_group(driver)
    procedure_qna = get_procedure_qna(driver)
    downtime_info = safe_section_text(driver, SEL_DOWNTIME_INFO, sep=" ")
    side_effect_info = get_side_effect_info(driver)

    row = {
        "event_url": url,
        "대분류": depth1,
        "중분류": depth2,
        "소분류": depth3,
        "treatment_name": treatment_name,
        "main_image_url": main_image_url,
        "rating": rating,
        "review_count": review_count,
        "original_price": original_price,
        "selling_price": selling_price,
        "hospital_name": hospital_name,
        "hospital_address": hospital_address,
        "treatment_description": treatment_description,
        "option_info": option_info,
        "target_group": target_group,
        "procedure_info": procedure_info,
        "procedure_qna": procedure_qna,
        "downtime_info": downtime_info,
        "side_effect_info": side_effect_info,
    }
    return row


def crawl_event_page_with_retry(driver, url, depth1=None, depth2=None, depth3=None):
    last_exc = None
    for attempt in range(1, MAX_RETRY + 1):
        try:
            return crawl_event_page(driver, url, depth1, depth2, depth3)
        except (TimeoutException, UnexpectedAlertPresentException) as e:
            last_exc = e
            try:
                alert = driver.switch_to.alert
                alert.dismiss()
            except NoAlertPresentException:
                pass
            time.sleep(2)
        except Exception as e:
            last_exc = e
            break
    raise last_exc


# =========================
# 5. 메인 실행부
# =========================

def main():
    if not os.path.exists(EXCEL_PATH):
        raise FileNotFoundError(f"{EXCEL_PATH} 파일을 찾을 수 없습니다.")

    df = pd.read_excel(EXCEL_PATH)

    if "event_url" not in df.columns:
        raise ValueError("엑셀에 'event_url' 컬럼이 없습니다. 컬럼명을 확인해주세요.")

    total_rows = len(df)
    print(f"[정보] 전체 URL 수: {total_rows}개")

    if TEST_MODE:
        sample_n = min(TEST_SAMPLE_SIZE, total_rows)
        df_target = df.sample(n=sample_n, random_state=42).reset_index(drop=True)
        print(f"[테스트 모드] 랜덤 {sample_n}개 URL만 크롤링합니다.")
    else:
        if MAX_ROWS is not None:
            df_target = df.head(MAX_ROWS).reset_index(drop=True)
            print(f"[부분 실행] 상위 {len(df_target)}개 URL만 크롤링합니다.")
        else:
            df_target = df.reset_index(drop=True)
            print(f"[전체 실행] 전체 {len(df_target)}개 URL 크롤링합니다.")

    if len(df_target) == 0:
        print("⚠️ 크롤링할 URL이 없습니다.")
        return

    results = []

    if os.path.exists(OUTPUT_CSV):
        try:
            df_done = pd.read_csv(OUTPUT_CSV, encoding="utf-8-sig")
            if "event_url" in df_done.columns:
                done_urls = set(df_done["event_url"].dropna().astype(str))
                df_target = df_target[~df_target["event_url"].astype(str).isin(done_urls)]
                results.extend(df_done.to_dict("records"))
                print(
                    f"[재시작] 기존 수집 건수: {len(results)}개 "
                    f"(이번에 새로 수집할 URL: {len(df_target)}개)"
                )
        except Exception as e:
            print(f"[경고] 기존 결과 파일 읽기 실패. 새로 시작합니다. 에러: {e}")

    total_to_crawl = len(df_target)
    if total_to_crawl == 0:
        print("✅ 모든 대상 URL이 이미 수집되었습니다.")
        save_results(results, OUTPUT_CSV, final=True)
        return

    driver = init_driver()
    print(f"[실행 정보] 실제 크롤링할 URL 수: {total_to_crawl}개")

    try:
        for idx, row in enumerate(df_target.itertuples(index=False), start=1):
            url = getattr(row, "event_url")
            depth1 = getattr(row, "대분류", None) if "대분류" in df_target.columns else None
            depth2 = getattr(row, "중분류", None) if "중분류" in df_target.columns else None
            depth3 = getattr(row, "소분류", None) if "소분류" in df_target.columns else None

            url = str(url)
            print(f"[{idx}/{total_to_crawl}] 크롤링 중 → {url}")

            try:
                detail = crawl_event_page_with_retry(
                    driver, url, depth1=depth1, depth2=depth2, depth3=depth3
                )
                results.append(detail)
            except Exception as e:
                print(f"⚠️ URL 스킵: {url} ({type(e).__name__}: {e})")
                log_error_row(url, depth1, depth2, depth3, e)

            if idx % SAVE_INTERVAL == 0:
                save_results(results, OUTPUT_CSV, final=False)

            time.sleep(1.0)

    finally:
        driver.quit()

    save_results(results, OUTPUT_CSV, final=True)
    print(f"\n✅ 크롤링 종료. 최종 {len(results)}건 저장 → {OUTPUT_CSV}")
    print("✅ CSV / XLSX 저장 완료")


if __name__ == "__main__":
    try:
        main()
    except Exception as main_e:
        print(f"\n❌ 프로그램 메인 실행 중 치명적인 에러 발생: {main_e}")


[정보] 전체 URL 수: 4624개
[테스트 모드] 랜덤 100개 URL만 크롤링합니다.
[실행 정보] 실제 크롤링할 URL 수: 100개
[1/100] 크롤링 중 → https://www.yeoshin.co.kr/event/mobile/27300
[2/100] 크롤링 중 → https://www.yeoshin.co.kr/event/mobile/22251
[3/100] 크롤링 중 → https://www.yeoshin.co.kr/event/mobile/11119
[4/100] 크롤링 중 → https://www.yeoshin.co.kr/event/mobile/23298
[5/100] 크롤링 중 → https://www.yeoshin.co.kr/event/mobile/15888
[6/100] 크롤링 중 → https://www.yeoshin.co.kr/event/mobile/16728
[7/100] 크롤링 중 → https://www.yeoshin.co.kr/event/mobile/19423
[8/100] 크롤링 중 → https://www.yeoshin.co.kr/event/mobile/27525
[9/100] 크롤링 중 → https://www.yeoshin.co.kr/event/mobile/25078
[10/100] 크롤링 중 → https://www.yeoshin.co.kr/event/mobile/24102
[중간 저장] 10건 → yeoti_detail_procedure_part_01.csv, yeoti_detail_procedure_part_01.xlsx
[11/100] 크롤링 중 → https://www.yeoshin.co.kr/event/mobile/20401
[12/100] 크롤링 중 → https://www.yeoshin.co.kr/event/mobile/24443
[13/100] 크롤링 중 → https://www.yeoshin.co.kr/event/mobile/19204
[14/100] 크롤링 중 → https://www.yeoshin.c